# ResnNet50

In [1]:
import pickle 
import numpy as np
from tensorflow.keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model, Sequential
from keras.layers import Input
import matplotlib.pyplot as plt
from glob import glob
import os
from keras.applications.resnet50 import ResNet50
import tensorflow as tf
from glob import glob
import json
import cv2
import json
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
import time

In [2]:

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [26]:
feature_model = ResNet50(include_top=True)

extractor = Model(inputs = feature_model.input,outputs = feature_model.layers[-2].output)

model = tf.keras.models.load_model('../Logs/ResNet50_BNG9k/logs/caption_model_RMSprop_BidirectionalLstm_512_256_512_24.06.2021-04_23/caption_model_RMSprop_BidirectionalLstm_512_256_512_24.06.2021-04_23.hdf5')


In [27]:
with open('../Data/BNG_9k/inv_dict.json') as f:
    inv_dict = json.load(f)

with open('../Data/BNG_9k/word_count.json') as f:
    word_count = json.load(f)
    
with open("../Data/BNG_9k/captions_dict.json") as f:
    captions_dict = json.loads(f.read())
    
with open("../Data/test/rename/ref_captions.txt",'r',encoding='utf-8') as f:
    ref_caption = f.readlines()

In [28]:
def getImg(test_img_path):
    

    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    test_img = cv2.resize(test_img, (224,224))

    test_img = np.reshape(test_img, (1,224,224,3))
    
    return test_img

In [29]:

MAX_LEN=39
gen_caption = []
gen_time = []
extract_time = []
images2 = glob('../Data/test/rename/'+'*.jpg')

for img_path in images2:
    
    start1 = time.time()
    test_feature = extractor.predict(getImg(img_path)).reshape(1,2048)
    end1 = time.time()
    
    test_img_path = img_path
    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    start2 = time.time()
    text_inp = ['startofseq']
    count = 0
    caption = ''
    while count < 25:
        count += 1

        encoded = []
        for i in text_inp:
            encoded.append(word_count[i])

        encoded = [encoded]
        encoded = pad_sequences(encoded, padding='post', truncating='post', maxlen=MAX_LEN)


        prediction = np.argmax(model.predict([test_feature, encoded]))
        sampled_word = inv_dict[str(prediction)]  
            
        if sampled_word == 'endofseq':
            break
        else:
            caption = caption + ' ' + sampled_word

        text_inp.append(sampled_word)
    end2 = time.time()
    gen_caption.append(caption.strip())
    extract_time.append(end1-start1)
    gen_time.append(end2-start2)

In [30]:
print('Average generation time',np.mean(gen_time)*1000,'ms')
print('Average extraction time',np.mean(extract_time)*1000,'ms')

Average generation time 384.769354111109 ms
Average extraction time 101.51402461223114 ms


In [31]:
ref_caption_processed = []
for line in ref_caption:
    ref_caption_processed.append(line.strip().split('%')[-1])
    
embedded_ref_caption = []
for text in ref_caption_processed:
    embedded_ref_caption.append([str(word_count[i]) for i in text.split()])
    
embedded_gen_caption = []
for text in gen_caption:
    embedded_gen_caption.append([str(word_count[i]) for i in text.split()])

In [32]:
BLEU_1 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    BLEU_1.append(corpus_bleu(ref,gen, weights=(1, 0, 0.0,0.0)))

print('BLEU 1 score:',np.mean(BLEU_1)*100,'%')

BLEU_2 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    BLEU_2.append(corpus_bleu(ref,gen, weights=(0.5, 0.5, 0.0,0.0)))

print('BLEU 2 score:',np.mean(BLEU_2)*100,'%')

BLEU_3 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    BLEU_3.append(corpus_bleu(ref,gen, weights=(0.3, 0.3, 0.3,0.0)))

print('BLEU 3 score:',np.mean(BLEU_3)*100,'%')


BLEU_4 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    BLEU_4.append(corpus_bleu(ref,gen, weights=(0.25, 0.25, 0.25, 0.25)))

print('BLEU 4 score:',np.mean(BLEU_4)*100,'%')

BLEU 1 score: 90.85546737459893 %
BLEU 2 score: 1.4208453866725605e-152 %
BLEU 2 score: 2.486899892182575e-183 %
BLEU 2 score: 1.777744703793022e-229 %


# Xception

In [33]:
import pickle 
import numpy as np
from tensorflow.keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model, Sequential
from keras.layers import Input
import matplotlib.pyplot as plt
from glob import glob
import os
from tensorflow.keras.applications import Xception
import tensorflow as tf
from glob import glob
import json
import cv2
import json
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
import time

In [11]:

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [12]:
feature_model = Xception(include_top=True)

extractor = Model(inputs = feature_model.input,outputs = feature_model.layers[-2].output)
model = tf.keras.models.load_model('../Logs/Xception_BNG9k/logs/caption_model_RMSprop_BidirectionalLstm_512_256_512_24.06.2021-22_45/caption_model_RMSprop_BidirectionalLstm_512_256_512_24.06.2021-22_45.hdf5')


In [13]:
with open('../Data/BNG_9k/inv_dict.json') as f:
    inv_dict = json.load(f)

with open('../Data/BNG_9k/word_count.json') as f:
    word_count = json.load(f)
    
with open("../Data/BNG_9k/captions_dict.json") as f:
    captions_dict = json.loads(f.read())
    
with open("../Data/test/rename/ref_captions.txt",'r',encoding='utf-8') as f:
    ref_caption = f.readlines()

In [14]:
def getImg(test_img_path):
    

    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    test_img = cv2.resize(test_img, (299,299))

    test_img = np.reshape(test_img, (1,299,299,3))
    
    return test_img

In [15]:

MAX_LEN=39
gen_caption = []
gen_time = []
extract_time = []
images2 = glob('../Data/test/rename/'+'*.jpg')

for img_path in images2:
    
    start1 = time.time()
    test_feature = extractor.predict(getImg(img_path)).reshape(1,2048)
    end1 = time.time()
    
    test_img_path = img_path
    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    start2 = time.time()
    text_inp = ['startofseq']
    count = 0
    caption = ''
    while count < 25:
        count += 1

        encoded = []
        for i in text_inp:
            encoded.append(word_count[i])

        encoded = [encoded]
        encoded = pad_sequences(encoded, padding='post', truncating='post', maxlen=MAX_LEN)


        prediction = np.argmax(model.predict([test_feature, encoded]))
        sampled_word = inv_dict[str(prediction)]  
            
        if sampled_word == 'endofseq':
            break
        else:
            caption = caption + ' ' + sampled_word

        text_inp.append(sampled_word)
    end2 = time.time()
    gen_caption.append(caption.strip())
    extract_time.append(end1-start1)
    gen_time.append(end2-start2)

In [16]:
print('Average generation time',np.mean(gen_time)*1000,'ms')
print('Average extraction time',np.mean(extract_time)*1000,'ms')

Average generation time 460.67705826881604 ms
Average extraction time 135.96720573229666 ms


In [17]:
ref_caption_processed = []
for line in ref_caption:
    ref_caption_processed.append(line.strip().split('%')[-1])
    
embedded_ref_caption = []
for text in ref_caption_processed:
    embedded_ref_caption.append([str(word_count[i]) for i in text.split()])
    
embedded_gen_caption = []
for text in gen_caption:
    embedded_gen_caption.append([str(word_count[i]) for i in text.split()])

In [18]:
BLEU_1 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_1.append(corpus_bleu(ref,gen[:len(ref)], weights=(1, 0, 0.0,0.0)))
    except:
        continue
print('BLEU 1 score:',np.mean(BLEU_1)*100,'%')


BLEU_2 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_2.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.5, 0.5, 0.0,0.0)))
    except:
        continue
print('BLEU 2 score:',np.mean(BLEU_2)*100,'%')


BLEU_3 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_3.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.3, 0.3, 0.3, 0.0)))
    except:
        continue
print('BLEU 3 score:',np.mean(BLEU_3)*100,'%')


BLEU_4 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_4.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.25, 0.25, 0.25, 0.25)))
    except:
        continue
print('BLEU 2 score:',np.mean(BLEU_4)*100,'%')

BLEU 1 score: 50.96425980147799 %
BLEU 2 score: 1.0078212141343556e-152 %
BLEU 3 score: 1.9948385178362915e-183 %
BLEU 2 score: 1.4748232207410426e-229 %


# VGG16

In [34]:
import pickle 
import numpy as np
from tensorflow.keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model, Sequential
from keras.layers import Input
import matplotlib.pyplot as plt
from glob import glob
import os
from keras.applications.vgg16 import VGG16
import tensorflow as tf
from glob import glob
import json
import cv2
import json
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
import time

In [35]:

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [36]:
feature_model = VGG16(include_top=True)

extractor = Model(inputs = feature_model.input,outputs = feature_model.layers[-2].output)
model = tf.keras.models.load_model('../Logs/VGG16_BNG9k/logs/caption_model_RMSprop_BidirectionalLstm_512_256_512_26.06.2021-15_26/caption_model_RMSprop_BidirectionalLstm_512_256_512_26.06.2021-15_26.hdf5')

In [37]:
with open('../Data/BNG_9k/inv_dict.json') as f:
    inv_dict = json.load(f)

with open('../Data/BNG_9k/word_count.json') as f:
    word_count = json.load(f)
    
with open("../Data/BNG_9k/captions_dict.json") as f:
    captions_dict = json.loads(f.read())
    
with open("../Data/test/rename/ref_captions.txt",'r',encoding='utf-8') as f:
    ref_caption = f.readlines()

In [38]:
def getImg(test_img_path):
    

    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    test_img = cv2.resize(test_img, (224,224))

    test_img = np.reshape(test_img, (1,224,224,3))
    
    return test_img

In [39]:

MAX_LEN=39
gen_caption = []
gen_time = []
extract_time = []
images2 = glob('../Data/test/rename/'+'*.jpg')

for img_path in images2:
    
    start1 = time.time()
    test_feature = extractor.predict(getImg(img_path)).reshape(1,4096)
    end1 = time.time()
    
    test_img_path = img_path
    test_img = cv2.imread(test_img_path)
    test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    start2 = time.time()
    text_inp = ['startofseq']
    count = 0
    caption = ''
    while count < 25:
        count += 1

        encoded = []
        for i in text_inp:
            encoded.append(word_count[i])

        encoded = [encoded]
        encoded = pad_sequences(encoded, padding='post', truncating='post', maxlen=MAX_LEN)


        prediction = np.argmax(model.predict([test_feature, encoded]))
        sampled_word = inv_dict[str(prediction)]  
            
        if sampled_word == 'endofseq':
            break
        else:
            caption = caption + ' ' + sampled_word

        text_inp.append(sampled_word)
    end2 = time.time()
    gen_caption.append(caption.strip())
    extract_time.append(end1-start1)
    gen_time.append(end2-start2)

In [40]:
print('Average generation time',np.mean(gen_time)*1000,'ms')
print('Average extraction time',np.mean(extract_time)*1000,'ms')

Average generation time 442.7652481274727 ms
Average extraction time 131.5593108152732 ms


In [41]:
ref_caption_processed = []
for line in ref_caption:
    ref_caption_processed.append(line.strip().split('%')[-1])
    
embedded_ref_caption = []
for text in ref_caption_processed:
    embedded_ref_caption.append([str(word_count[i]) for i in text.split()])
    
embedded_gen_caption = []
for text in gen_caption:
    embedded_gen_caption.append([str(word_count[i]) for i in text.split()])

In [42]:
BLEU_1 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_1.append(corpus_bleu(ref,gen[:len(ref)], weights=(1, 0, 0.0,0.0)))
    except:
        continue
print('BLEU 1 score:',np.mean(BLEU_1)*100,'%')


BLEU_2 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_2.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.5, 0.5, 0.0,0.0)))
    except:
        continue
print('BLEU 2 score:',np.mean(BLEU_2)*100,'%')


BLEU_3 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_3.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.3, 0.3, 0.3, 0.0)))
    except:
        continue
print('BLEU 3 score:',np.mean(BLEU_3)*100,'%')


BLEU_4 = []
for ref , gen in zip(embedded_ref_caption, embedded_gen_caption):
    try:
        BLEU_4.append(corpus_bleu(ref,gen[:len(ref)], weights=(0.25, 0.25, 0.25, 0.25)))
    except:
        continue
print('BLEU 2 score:',np.mean(BLEU_4)*100,'%')

BLEU 1 score: 54.49484450556393 %
BLEU 2 score: 1.057690054252067e-152 %
BLEU 3 score: 2.062099510702824e-183 %
BLEU 2 score: 1.5175274910911872e-229 %
